In [1]:

!pip install torch

import torch
import torch.nn as nn
import numpy as np

# Dataset
text = """machine learning models learn patterns from data.
sequence models process data step by step.
recurrent neural networks are designed for sequential tasks.
rnn models maintain hidden states across time steps.

long short term memory networks solve long dependency problems.
lstm uses gates to control information flow.
gru models simplify the lstm architecture.
sequence prediction is useful in many applications.

language modeling predicts the next word in a sentence.
speech recognition processes audio sequences.
time series forecasting predicts future values.
music generation creates new melodies.

generative models learn probability distributions.
they generate new samples similar to training data.
sequence generation is widely used in artificial intelligence.
deep learning improves sequence modeling performance.
"""

# Preprocessing
words = text.split()
vocab = list(set(words))
word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}

# Create sequences
seq_length = 5
inputs = []
targets = []

for i in range(len(words) - seq_length):
    inputs.append([word2idx[w] for w in words[i:i+seq_length]])
    targets.append(word2idx[words[i+seq_length]])

X = torch.tensor(inputs)
y = torch.tensor(targets)

# Model
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_size=32, hidden_size=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

model = LSTMModel(len(vocab))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training
for epoch in range(100):
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

# Generate text
def generate(seed, length=10):
    model.eval()
    words_list = seed.split()

    for _ in range(length):
        inp = torch.tensor([[word2idx.get(w,0) for w in words_list[-seq_length:]]])
        out = model(inp)
        pred = torch.argmax(out).item()
        words_list.append(idx2word[pred])

    return " ".join(words_list)

print(generate("machine learning models learn"))

Epoch 0, Loss: 4.484795570373535
Epoch 20, Loss: 0.22652386128902435
Epoch 40, Loss: 0.007857569493353367
Epoch 60, Loss: 0.0029520979151129723
Epoch 80, Loss: 0.0020813208539038897
machine learning models learn probability distributions. they generate new samples similar to training data.


In [2]:
import torch
import torch.nn as nn

# Simple Transformer Model
class TransformerModel(nn.Module):
    def __init__(self, vocab_size, embed_size=32, num_heads=2, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.pos_encoder = nn.Parameter(torch.randn(1, 100, embed_size))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_size, nhead=num_heads
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc = nn.Linear(embed_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x) + self.pos_encoder[:, :x.size(1), :]
        x = self.transformer(x)
        x = self.fc(x[:, -1, :])
        return x

model_t = TransformerModel(len(vocab))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_t.parameters(), lr=0.01)

# Training
for epoch in range(100):
    optimizer.zero_grad()
    output = model_t(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

# Generate text
def generate_transformer(seed, length=10):
    model_t.eval()
    words_list = seed.split()

    for _ in range(length):
        inp = torch.tensor([[word2idx.get(w,0) for w in words_list[-seq_length:]]])
        out = model_t(inp)
        pred = torch.argmax(out).item()
        words_list.append(idx2word[pred])

    return " ".join(words_list)

print(generate_transformer("deep learning improves"))

/tmp/ipykernel_1383/1333161605.py:14: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)


Epoch 0, Loss: 4.651882648468018
Epoch 20, Loss: 0.8267844319343567
Epoch 40, Loss: 0.32645419239997864
Epoch 60, Loss: 0.3227725923061371
Epoch 80, Loss: 0.3097364008426666
deep learning improves sequence generation is useful in many applications. language modeling performance.
